# Semantic Drift — Colab Pipeline

Runs the GPU-dependent parts of the project: InstructPix2Pix baseline edit chains and a SAM segmentation smoke test.

**Before running:** Runtime → Change runtime type → T4 GPU (free tier).

**Getting the project onto Colab** (`data/raw_images/` and `data/edit_instructions.json` are gitignored, so a plain `git clone` alone won't bring the dataset):
- **Option A (simplest):** on your laptop, zip the whole `Implementation/` folder, then use the folder icon on the left of Colab to upload the zip to `/content/Implementation.zip`. Run the cell below to unzip it.
- **Option B:** if you've pushed the repo to GitHub, `git clone` it, then separately upload `data/raw_images/` and `data/edit_instructions.json` into the cloned folder (they won't come from git).

In [ ]:
!pip install -q diffusers transformers accelerate segment-anything opencv-python pyyaml

In [ ]:
import zipfile
import os

zip_path = "/content/Implementation.zip"
extract_to = "/content"

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    print("Extracted to", extract_to)
else:
    print(f"{zip_path} not found — upload it (Option A) or git clone + upload data/ manually (Option B),")
    print("then update PROJECT_ROOT in the next cell if your folder lands somewhere else.")

PROJECT_ROOT = "/content/Implementation"

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU — set Runtime > Change runtime type > GPU")

## Load the dataset

Validates that every image referenced in `edit_instructions.json` actually exists in `raw_images/`.

In [ ]:
from pathlib import Path
from src.data_loader import load_edit_chains

DATA_DIR = Path(PROJECT_ROOT) / "data"
chains = load_edit_chains(str(DATA_DIR / "edit_instructions.json"), str(DATA_DIR / "raw_images"))
print(f"Loaded {len(chains)} chains over {len(set(c['image_id'] for c in chains))} images")

## SAM smoke test

Quick sanity check that segmentation works before Day 14-15's full drift-scoring integration. Downloads the SAM `vit_b` checkpoint (~375MB) the first time it's called.

In [ ]:
from PIL import Image
from src.segment import segment_image

sample_id = chains[0]["image_id"]
sample_image = Image.open(DATA_DIR / "raw_images" / sample_id).convert("RGB")
regions = segment_image(sample_image)
print(f"SAM found {len(regions)} regions in {sample_id}")

## Run baseline edit chains (no mitigation)

Each image is resized to 512x512 (standard for this Stable-Diffusion-based pipeline), then every instruction in its chain is applied in sequence, saving every intermediate step to `results/baseline/`. Safe to re-run after a Colab disconnect — already-completed chains are skipped.

In [ ]:
from src.edit_runner import run_edit_chain

RESULTS_DIR = Path(PROJECT_ROOT) / "results" / "baseline"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for i, chain in enumerate(chains, 1):
    image_id = chain["image_id"]
    chain_type = chain["chain_type"]
    stem = Path(image_id).stem
    out_dir = RESULTS_DIR / f"{stem}_{chain_type}"

    if out_dir.exists() and len(list(out_dir.glob("step*.png"))) == len(chain["instructions"]) + 1:
        continue  # already done

    out_dir.mkdir(parents=True, exist_ok=True)
    image = Image.open(DATA_DIR / "raw_images" / image_id).convert("RGB").resize((512, 512))
    image.save(out_dir / "step0_original.png")

    outputs = run_edit_chain(image, chain["instructions"])
    for step, edited in enumerate(outputs, 1):
        edited.save(out_dir / f"step{step}.png")

    print(f"[{i}/{len(chains)}] {stem} ({chain_type}) done")

print("All baseline chains complete.")

## Next steps

Download `results/baseline/` back to your laptop (zip it first) before the Colab session ends — the runtime and its disk are wiped on disconnect. Day 14-15 (baseline Drift Score collection) runs `drift_score.py` over these saved outputs.